<a href="https://colab.research.google.com/github/sarahlimadev/sarahlimadev/blob/main/code_examples/2025-12/assistente_vendas_apigemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Criando um Assistente de Vendas usando API Gemini
**Autor:** Sarah P. Lima | **Data:** 19 Dec 2025

Neste notebook, vamos implementar um assistente de vendas inteligente capaz de registrar pedidos através de conversas naturais, utilizando o recurso de **Function Calling** da API Gemini.

## O que vamos construir?
Um chat interativo onde o modelo de IA identifica quando o usuário quer fazer um pedido, extrai os dados (produto e quantidade) e executa uma função Python para salvar esses dados.

## **Célula 2 (Código / Instalação)**
Instalação das bibliotecas

In [ ]:
# Instalação da biblioteca oficial do Google Generative AI
!pip install -q -U google-generativeai

## Configuração do Ambiente

No artigo original, utilizamos um arquivo `.env` para proteger a chave de API. Como estamos no Google Colab, utilizaremos o gerenciador de **Segredos** (ícone da chave 🔑 na barra lateral esquerda).

1. Clique na chave na barra lateral.
2. Adicione um novo segredo chamado `GOOGLE_API_KEY` com o valor da sua chave.
3. Ative a opção "Notebook access" para este segredo.

## **Célula 4 (Código / Configuração)**
Código de configuração adaptado para Colab.

In [ ]:
import google.generativeai as genai
from google.colab import userdata

try:
    # Obtém a chave dos Segredos do Colab
    api_key = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=api_key)
    print("API Key configurada com sucesso!")
except Exception as e:
    print("Erro: Certifique-se de ter criado o segredo 'GOOGLE_API_KEY' na barra lateral.")

## Definindo a Função (Forma Prática)

Conforme explicado no artigo, usaremos a **Forma Prática** de function calling. A biblioteca `google-generativeai` lerá automaticamente os *Type Hints* e a *Docstring* da nossa função para criar a estrutura JSON necessária para o modelo.

## **Célula 6 (Código / Função)**
A função exata do artigo.

In [ ]:
# Lista para armazenar pedidos (simulando um banco de dados)
pedidos = []

def registrar_pedido(cliente: str, produto: str, quantidade: int) -> str:
    """Registra um novo pedido de venda

    Args:
        cliente: Nome do cliente que está fazendo o pedido
        produto: Nome do produto solicitado
        quantidade: Quantidade de unidades do produto

    Returns:
        Mensagem de confirmação do pedido
    """
    # Cria o objeto do pedido
    pedido = {
        "cliente": cliente,
        "produto": produto,
        "quantidade": quantidade
    }

    # Salva na lista global
    pedidos.append(pedido)

    # Retorna o feedback para o modelo usar na resposta
    return f"Pedido registrado com sucesso! Cliente: {cliente}, Produto: {produto}, Quantidade: {quantidade}"

print("Função 'registrar_pedido' definida.")

## Inicializando o Modelo e o Chat

Agora configuramos o modelo passando nossa função no parâmetro `tools`. Habilitamos também o `enable_automatic_function_calling` para que a API execute a função Python e gere a resposta final automaticamente.

## **Célula 8 (Código / Execução)**
O loop principal do chat.

In [ ]:
# Configurar o modelo
model = genai.GenerativeModel(
    model_name="gemini-2.5-flash", # Ajuste a versão se necessário
    tools=[registrar_pedido]      # Passamos a função aqui
)

# Iniciar o chat com histórico automático e chamada de função automática
chat = model.start_chat(enable_automatic_function_calling=True)

print("--- Assistente de Vendas Iniciado! ---")
print("Dica: Tente dizer 'Quero comprar 5 notebooks' ou 'Meu nome é [Seu Nome] e quero 2 mouses'.")
print("Digite 'sair' para encerrar.\n")

# Loop principal da conversa
while True:
    try:
        # Input do usuário no Colab
        mensagem_usuario = input("Você: ")

        # Verificar se quer sair
        if mensagem_usuario.lower() in ["sair", "exit", "parar"]:
            print("\n--- Resumo dos pedidos registrados nesta sessão ---")
            if not pedidos:
                print("Nenhum pedido registrado.")
            else:
                for i, pedido in enumerate(pedidos, 1):
                    print(f"{i}. {pedido['cliente']} - {pedido['quantidade']}x {pedido['produto']}")
            break

        # Enviar mensagem para o Gemini
        response = chat.send_message(mensagem_usuario)

        # Exibir resposta do Assistente
        print(f"Assistente: {response.text}\n")

    except Exception as e:
        print(f"Ocorreu um erro: {e}")